# 🏏 T20 World Cup 2026 — Exploratory Data Analysis

This notebook explores the cleaned match data to uncover patterns in:
- Score distributions and margins
- Chasing vs defending outcomes
- Venue analysis
- Toss impact
- Team performance profiles
- Feature correlations with match outcomes

In [12]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.config import DATA_PROCESSED, TEAM_CODES, VENUES
from src.utils import load_dataframe

# Style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('viridis')
pd.set_option('display.max_columns', 50)

print('Libraries loaded ✅')

Libraries loaded ✅


## 1. Load Data

In [13]:
# Load cleaned match data
matches = load_dataframe(DATA_PROCESSED / 'matches.csv')
features = load_dataframe(DATA_PROCESSED / 'match_features.csv')
batting = load_dataframe(DATA_PROCESSED / 'player_batting.csv')
bowling = load_dataframe(DATA_PROCESSED / 'player_bowling.csv')

print(f'Matches: {len(matches) if matches is not None else 0} rows')
print(f'Features: {len(features) if features is not None else 0} rows')
print(f'Batting: {len(batting) if batting is not None else 0} rows')
print(f'Bowling: {len(bowling) if bowling is not None else 0} rows')

2026-03-06 09:18:10 │ t20wc2026          │ ERROR   │ File not found: D:\DataCamp\T20 World Cup 2026 data science project\data_processed\matches.csv
2026-03-06 09:18:10 │ t20wc2026          │ ERROR   │ File not found: D:\DataCamp\T20 World Cup 2026 data science project\data_processed\match_features.csv
2026-03-06 09:18:10 │ t20wc2026          │ ERROR   │ File not found: D:\DataCamp\T20 World Cup 2026 data science project\data_processed\player_batting.csv
2026-03-06 09:18:10 │ t20wc2026          │ ERROR   │ File not found: D:\DataCamp\T20 World Cup 2026 data science project\data_processed\player_bowling.csv


Matches: 0 rows
Features: 0 rows
Batting: 0 rows
Bowling: 0 rows


In [14]:
if matches is not None:
    print('Match columns:', matches.columns.tolist())
    matches.head()

## 2. Score Distributions

In [15]:
if matches is not None and 'innings1_runs' in matches.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # First innings score distribution
    axes[0].hist(matches['innings1_runs'].dropna(), bins=15, color='#6366f1', 
                 edgecolor='white', alpha=0.8)
    axes[0].set_title('1st Innings Score Distribution', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Runs')
    axes[0].set_ylabel('Frequency')
    axes[0].axvline(matches['innings1_runs'].mean(), color='#f59e0b', 
                    linestyle='--', label=f"Mean: {matches['innings1_runs'].mean():.0f}")
    axes[0].legend()
    
    # Second innings
    axes[1].hist(matches['innings2_runs'].dropna(), bins=15, color='#a855f7', 
                 edgecolor='white', alpha=0.8)
    axes[1].set_title('2nd Innings Score Distribution', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Runs')
    axes[1].set_ylabel('Frequency')
    axes[1].axvline(matches['innings2_runs'].mean(), color='#f59e0b', 
                    linestyle='--', label=f"Mean: {matches['innings2_runs'].mean():.0f}")
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig(str(DATA_PROCESSED / 'plots' / 'score_distributions.png'), dpi=150)
    plt.show()
else:
    print('⚠️ Match data not available. Run scrapers + cleaner first.')

⚠️ Match data not available. Run scrapers + cleaner first.


## 3. Chasing vs Defending

In [16]:
if matches is not None and 'winner' in matches.columns:
    # Who wins more — batting first or chasing?
    bat_first_wins = len(matches[matches['winner'] == matches['team1']])
    chase_wins = len(matches[matches['winner'] == matches['team2']])
    total = bat_first_wins + chase_wins
    
    fig = go.Figure(data=[go.Pie(
        labels=['Batting First Won', 'Chasing Won'],
        values=[bat_first_wins, chase_wins],
        hole=0.5,
        marker=dict(colors=['#6366f1', '#f59e0b']),
        textinfo='label+percent',
    )])
    fig.update_layout(
        title='Batting First vs Chasing — Win Distribution',
        font=dict(family='Inter'),
        height=400,
    )
    fig.show()
    
    print(f'Batting first wins: {bat_first_wins} ({bat_first_wins/total:.0%})')
    print(f'Chasing wins: {chase_wins} ({chase_wins/total:.0%})')
else:
    print('⚠️ Match data not available.')

⚠️ Match data not available.


## 4. Venue Analysis

In [17]:
if matches is not None and 'venue' in matches.columns:
    venue_stats = matches.groupby('venue').agg(
        matches_played=('match_id', 'count'),
        avg_1st_innings=('innings1_runs', 'mean'),
        avg_2nd_innings=('innings2_runs', 'mean'),
    ).reset_index().sort_values('avg_1st_innings', ascending=False)
    
    if not venue_stats.empty:
        fig = px.bar(
            venue_stats, x='venue', y='avg_1st_innings',
            color='avg_1st_innings',
            color_continuous_scale='Viridis',
            title='Average 1st Innings Score by Venue',
            labels={'avg_1st_innings': 'Avg Score', 'venue': 'Venue'},
        )
        fig.update_layout(xaxis_tickangle=-45, height=500)
        fig.show()
        
        display(venue_stats.round(1))
else:
    print('⚠️ Venue data not available.')

⚠️ Venue data not available.


## 5. Team Win Rates

In [18]:
if matches is not None and 'winner' in matches.columns:
    all_teams = set(matches['team1'].unique()) | set(matches['team2'].unique())
    team_records = []
    
    for team in all_teams:
        if not team or pd.isna(team):
            continue
        played = len(matches[(matches['team1'] == team) | (matches['team2'] == team)])
        won = len(matches[matches['winner'] == team])
        team_records.append({
            'Team': TEAM_CODES.get(team, team),
            'Played': played,
            'Won': won,
            'Win Rate': won / played if played > 0 else 0
        })
    
    team_df = pd.DataFrame(team_records).sort_values('Win Rate', ascending=False)
    
    fig = px.bar(
        team_df, x='Team', y='Win Rate',
        color='Win Rate',
        color_continuous_scale=['#ef4444', '#f59e0b', '#22c55e'],
        title='Team Win Rates — T20 World Cup 2026',
        text='Won',
    )
    fig.update_layout(height=500)
    fig.show()
else:
    print('⚠️ Match data not available.')

⚠️ Match data not available.


## 6. Feature Correlations

In [19]:
if features is not None and 'team1_won' in features.columns:
    # Select numeric features for correlation
    numeric_cols = features.select_dtypes(include=[np.number]).columns.tolist()
    
    # Correlation with target
    corr_with_target = features[numeric_cols].corr()['team1_won'].drop('team1_won').sort_values(ascending=False)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = ['#22c55e' if v > 0 else '#ef4444' for v in corr_with_target.values]
    corr_with_target.plot(kind='barh', ax=ax, color=colors)
    ax.set_title('Feature Correlation with Match Outcome', fontsize=14, fontweight='bold')
    ax.set_xlabel('Correlation with Team 1 Winning')
    ax.axvline(0, color='gray', linewidth=0.5)
    plt.tight_layout()
    plt.savefig(str(DATA_PROCESSED / 'plots' / 'feature_correlations.png'), dpi=150)
    plt.show()
else:
    print('⚠️ Feature data not available. Run feature engineer first.')

⚠️ Feature data not available. Run feature engineer first.


## 7. Top Performers

In [20]:
if batting is not None and not batting.empty:
    top_scorers = batting.groupby('player').agg(
        total_runs=('runs', 'sum'),
        total_balls=('balls', 'sum'),
        fours=('fours', 'sum'),
        sixes=('sixes', 'sum'),
        innings=('runs', 'count'),
    ).reset_index()
    top_scorers['strike_rate'] = (top_scorers['total_runs'] / top_scorers['total_balls'] * 100).round(1)
    top_scorers = top_scorers.sort_values('total_runs', ascending=False).head(15)
    
    fig = px.bar(
        top_scorers, x='player', y='total_runs',
        color='strike_rate',
        color_continuous_scale='YlOrRd',
        title='Top 15 Run Scorers',
        text='total_runs',
        labels={'total_runs': 'Runs', 'player': 'Player', 'strike_rate': 'SR'},
    )
    fig.update_layout(xaxis_tickangle=-45, height=500)
    fig.show()
else:
    print('⚠️ Batting data not available.')

⚠️ Batting data not available.


In [21]:
if bowling is not None and not bowling.empty:
    top_bowlers = bowling.groupby('player').agg(
        total_wickets=('wickets', 'sum'),
        total_runs=('runs', 'sum'),
        total_overs=('overs', 'sum'),
        innings=('wickets', 'count'),
    ).reset_index()
    top_bowlers['economy'] = (top_bowlers['total_runs'] / top_bowlers['total_overs']).round(2)
    top_bowlers = top_bowlers.sort_values('total_wickets', ascending=False).head(15)
    
    fig = px.bar(
        top_bowlers, x='player', y='total_wickets',
        color='economy',
        color_continuous_scale='RdYlGn_r',
        title='Top 15 Wicket Takers',
        text='total_wickets',
        labels={'total_wickets': 'Wickets', 'player': 'Player', 'economy': 'Econ'},
    )
    fig.update_layout(xaxis_tickangle=-45, height=500)
    fig.show()
else:
    print('⚠️ Bowling data not available.')

⚠️ Bowling data not available.


## 8. Summary Statistics

In [ ]:
if matches is not None:
    print('=' * 60)
    print('T20 WORLD CUP 2026 — SUMMARY')
    print('=' * 60)
    print(f'Total matches:          {len(matches)}')
    print(f'Venues:                 {matches["venue"].nunique()}')
    print(f'Teams:                  {len(set(matches["team1"].unique()) | set(matches["team2"].unique()))}')
    print(f'Avg 1st innings score:  {matches["innings1_runs"].mean():.1f}')
    print(f'Avg 2nd innings score:  {matches["innings2_runs"].mean():.1f}')
    print(f'Highest score:          {matches["innings1_runs"].max():.0f}')
    print(f'Lowest score:           {matches["innings1_runs"].min():.0f}')
    print('=' * 60)
else:
    print('⚠️ Run the data pipeline first to generate data for EDA.')
    print()
    print('Commands:')
    print('  python src/scrapers/espn_scraper.py')
    print('  python src/scrapers/cricbuzz_scraper.py')
    print('  python src/scrapers/icc_scraper.py')
    print('  python src/processing/cleaner.py')
    print('  python src/processing/features.py')

⚠️ Run the data pipeline first to generate data for EDA.

Commands:
  python src/scrapers/espn_scraper.py
  python src/scrapers/cricbuzz_scraper.py
  python src/scrapers/icc_scraper.py
  python src/processing/cleaner.py
  python src/processing/features.py


: 